In [ ]:
# Download Final_Dataset.zip separately and place it in the project directory.
# The dataset is not included in this repository because of its size.
# Unzip the folder
!unzip "Final_Dataset.zip"

In [ ]:
# -----------------------------
# Step 1: Load Dataset
# -----------------------------
import os
import cv2
import numpy as np

IMG_SIZE = 128

def load_data(data_dir):

    data = []
    labels = []

    for label in ["Glioma",  "Healthy", "Other_tumor"]:

        path = os.path.join( data_dir, label )

        if label == "Healthy":
            class_num = 0
        elif label == "Glioma":
            class_num = 1
        else:  # Other_tumor
            class_num = 2

        for img in os.listdir(path):

            try:

                img_path = os.path.join( path, img )

                img_array = cv2.imread( img_path, cv2.IMREAD_GRAYSCALE)

                if img_array is None:
                    continue

                img_array = cv2.resize( img_array, (IMG_SIZE, IMG_SIZE) )

                data.append(img_array)

                labels.append(class_num)

            except:
                pass

    data = np.array(data).reshape(  -1, IMG_SIZE, IMG_SIZE, 1 ) / 255.0

    labels = np.array(labels)

    return data, labels

In [ ]:
# -----------------------------
# Step 2: Load + Shuffle
# -----------------------------
from sklearn.utils import shuffle
# Note: The 'os', 'cv2', and 'numpy' modules need to be imported in the cell
# where the 'load_data' function is defined (cell 'uMVmpIIPPPiS') for it to work correctly.

X, y = load_data("Final_Dataset")

X, y = shuffle(
    X,
    y,
    random_state=42
)

print("Total Images :", len(X))

In [ ]:
# -----------------------------
# Step 3: 70-30 Split
# -----------------------------
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(

    X,

    y,

    test_size=0.30,

    stratify=y,

    random_state=42

)

print("Train Images :", len(X_train))
print("Test Images  :", len(X_test))

In [ ]:
# step 4 split folders
import os
import shutil
from sklearn.model_selection import train_test_split
from google.colab import files

DATASET_PATH = "Final_Dataset"

train_folder = "train_data"
test_folder = "test_data"

# Create folders
os.makedirs("train_data/Glioma", exist_ok=True)
os.makedirs("train_data/Healthy", exist_ok=True)
os.makedirs("train_data/Other_tumor", exist_ok=True)

os.makedirs("test_data/Glioma", exist_ok=True)
os.makedirs("test_data/Healthy", exist_ok=True)
os.makedirs("test_data/Other_tumor", exist_ok=True)

glioma_paths = []
healthy_paths = []
other_paths = []

# Glioma Images
for img in os.listdir(
    os.path.join(
        DATASET_PATH,
        "Glioma"
    )
):
    glioma_paths.append(
        os.path.join(
            DATASET_PATH,
            "Glioma",
            img
        )
    )

# Healthy Images
for img in os.listdir(
    os.path.join(
        DATASET_PATH,
        "Healthy"
    )
):
    healthy_paths.append(
        os.path.join(
            DATASET_PATH,
            "Healthy",
            img
        )
    )

# Other Tumor Images
for img in os.listdir(
    os.path.join(
        DATASET_PATH,
        "Other_tumor"
    )
):
    other_paths.append(
        os.path.join(
            DATASET_PATH,
            "Other_tumor",
            img
        )
    )

# Split Data
glioma_train, glioma_test = train_test_split(
    glioma_paths,
    test_size=0.30,
    random_state=42
)

healthy_train, healthy_test = train_test_split(
    healthy_paths,
    test_size=0.30,
    random_state=42
)

other_train, other_test = train_test_split(
    other_paths,
    test_size=0.30,
    random_state=42
)

# Copy Train Data
for f in glioma_train:
    shutil.copy(f, "train_data/Glioma")

for f in healthy_train:
    shutil.copy(f, "train_data/Healthy")

for f in other_train:
    shutil.copy(f, "train_data/Other_tumor")

# Copy Test Data
for f in glioma_test:
    shutil.copy(f, "test_data/Glioma")

for f in healthy_test:
    shutil.copy(f, "test_data/Healthy")

for f in other_test:
    shutil.copy(f, "test_data/Other_tumor")

print("Train/Test Folders Created")

# Create Combined Folder
shutil.copytree(
    "train_data",
    "split_dataset/train_data",
    dirs_exist_ok=True
)

shutil.copytree(
    "test_data",
    "split_dataset/test_data",
    dirs_exist_ok=True
)

# Create ZIP
shutil.make_archive(
    "split_dataset",
    "zip",
    "split_dataset"
)

print("ZIP Created Successfully!")

# Download ZIP
files.download("split_dataset.zip")

In [ ]:
# -----------------------------
# Step 5: CNN Model
# -----------------------------
from tensorflow.keras import models, layers

model = models.Sequential([

    layers.Conv2D(
        32,
        (3,3),
        activation='relu',
        input_shape=(128,128,1)
    ),

    layers.MaxPooling2D(2,2),

    layers.Conv2D(
        64,
        (3,3),
        activation='relu'
    ),

    layers.MaxPooling2D(2,2),

    layers.Flatten(),

    layers.Dense(
        128,
        activation='relu'
    ),

    layers.Dropout(0.5),

    # 3 Classes:
    # Healthy = 0
    # Glioma = 1
    # Other_tumor = 2

    layers.Dense(
        3,
        activation='softmax'
    )

])

model.compile(

    optimizer='adam',

    loss='sparse_categorical_crossentropy',

    metrics=['accuracy']

)

model.summary()

In [ ]:
# -----------------------------
# Step 6: Train Model
# -----------------------------

history = model.fit(

    X_train,

    y_train,

    epochs=5,

    batch_size=32,

    validation_split=0.30

)


In [ ]:
# -----------------------------
# Step 7: Evaluate
# -----------------------------

loss, accuracy = model.evaluate( X_test, y_test,  verbose=0)

# Training Accuracy
train_accuracy = history.history['accuracy'][-1]
val_accuracy = history.history['val_accuracy'][-1]

print("\nTraining Accuracy :", train_accuracy * 100)
print("Test Accuracy     :", accuracy * 100)
print("Validation Accuracy :", val_accuracy * 100)

In [ ]:
# -----------------------------
# Step 8: Predictions
# -----------------------------

y_pred_probs = model.predict(

    X_test,

    verbose=0

)

y_pred = np.argmax(

    y_pred_probs,

    axis=1

)

In [ ]:
# -----------------------------
# Step 9: Confusion Matrix
# -----------------------------
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

cm = confusion_matrix(

    y_test,

    y_pred

)

plt.figure(figsize=(6,5))

sns.heatmap(

    cm,

    annot=True,

    fmt='d',

    cmap='Blues',

    xticklabels=['Healthy','Glioma','Other_tumor'],

    yticklabels=['Healthy','Glioma','Other_tumor']

)

plt.xlabel("Predicted")

plt.ylabel("Actual")

plt.title("Confusion Matrix")

plt.show()

In [ ]:
# -----------------------------
# Step 10: Classification Report
# -----------------------------
from sklearn.metrics import classification_report

print(

classification_report( y_test, y_pred,
   target_names=[ "Healthy", "Glioma", "Other_tumor" ]
))

In [ ]:
# -----------------------------
# Step 11: Accuracy Graph
# -----------------------------

plt.figure(figsize=(7,5))

plt.plot( history.history['accuracy'], marker='o', label='Train Accuracy')

plt.plot( history.history['val_accuracy'], marker='o', label='Validation Accuracy')

plt.title("Accuracy Graph")

plt.xlabel("Epoch")

plt.ylabel("Accuracy")

plt.legend()

plt.grid()

plt.show()

# -----------------------------
# Step 12: Loss Graph
# -----------------------------

plt.figure(figsize=(7,5))

plt.plot( history.history['loss'], marker='o',label='Train Loss')

plt.plot( history.history['val_loss'], marker='o', label='Validation Loss')

plt.title("Loss Graph")

plt.xlabel("Epoch")

plt.ylabel("Loss")

plt.legend()

plt.grid()

plt.show()

In [ ]:
train_acc=max(history.history['accuracy'])

val_acc=max(history.history['val_accuracy'])

test_acc=accuracy

labels=["Train","Validation","Test"]

values=[train_acc,val_acc,test_acc]

plt.figure(figsize=(6,5))

plt.bar(labels,values)

plt.ylabel("Accuracy")

plt.title("Accuracy Comparison")

plt.show()
train_loss=min(history.history['loss'])

val_loss=min(history.history['val_loss'])

plt.figure(figsize=(6,5))

plt.bar(["Train Loss","Val Loss"],[train_loss,val_loss])

plt.title("Loss Comparison")

plt.show()

In [ ]:
model.save("Glioma_Detect.keras")

print("✅ Detection Model Saved Successfully")
from google.colab import files

files.download("Glioma_Detect.keras")

In [ ]:
import tensorflow as tf
import keras

print("TensorFlow:", tf.__version__)
print("Keras:", keras.__version__)

In [ ]:
# -----------------------------
# Step 13: Upload & Predict
# (ZIP + NII + Image Support)
# -----------------------------

import nibabel as nib
import zipfile
import os
import shutil
from google.colab import files
import numpy as np
import cv2
import matplotlib.pyplot as plt

# -----------------------------
# Predict Single Image
# -----------------------------
def predict_from_image_array(img_array):

    img_array = cv2.resize(img_array, (128, 128))

    img_input = img_array.reshape(
        1, 128, 128, 1
    ) / 255.0

    pred = model.predict(
        img_input,
        verbose=0
    )

    class_id = np.argmax(pred)
    confidence = np.max(pred)

    return class_id, confidence


# -----------------------------
# Predict Images From Folder
# -----------------------------
def predict_all_slices_from_folder(folder_path):

    predictions = []

    for root, dirs, files_list in os.walk(folder_path):

        for file in files_list:

            if file.lower().endswith(
                ('.png', '.jpg', '.jpeg')
            ):

                img_path = os.path.join(
                    root,
                    file
                )

                img = cv2.imread(
                    img_path,
                    cv2.IMREAD_GRAYSCALE
                )

                if img is None:
                    continue

                if img.dtype == np.uint16:
                    img = (
                        img / 256
                    ).astype(np.uint8)

                pred = predict_from_image_array(img)

                predictions.append(pred)

    return predictions


# -----------------------------
# Predict ZIP / NII / IMAGE
# -----------------------------
def predict_all_slices(path):

    predictions = []

    # ZIP FILE
    if path.lower().endswith('.zip'):

        extract_folder = "temp_extracted"

        if os.path.exists(extract_folder):
            shutil.rmtree(extract_folder)

        with zipfile.ZipFile(path, 'r') as zip_ref:
            zip_ref.extractall(extract_folder)

        print("ZIP extracted successfully.")

        predictions = predict_all_slices_from_folder(
            extract_folder
        )

    # NII FILE
    elif path.lower().endswith(
        ('.nii', '.nii.gz')
    ):

        nii = nib.load(path)

        volume = nii.get_fdata()

        print(
            f"Total slices detected: {volume.shape[2]}"
        )

        for i in range(volume.shape[2]):

            slice_img = volume[:, :, i]

            slice_img = (
                slice_img - np.min(slice_img)
            )

            if np.max(slice_img) != 0:

                slice_img = (
                    slice_img /
                    np.max(slice_img)
                )

            slice_img = (
                slice_img * 255
            ).astype(np.uint8)

            pred = predict_from_image_array(
                slice_img
            )

            predictions.append(pred)

    # SINGLE IMAGE
    else:

        img = cv2.imread(
            path,
            cv2.IMREAD_GRAYSCALE
        )

        if img is None:

            print(
                "Invalid Image. Please upload a valid Brain MRI image."
            )

            return None

        if img.dtype == np.uint16:

            img = (
                img / 256
            ).astype(np.uint8)

        pred = predict_from_image_array(img)

        predictions.append(pred)

        plt.imshow(
            img,
            cmap='gray'
        )

        plt.title("Uploaded Image")
        plt.axis('off')
        plt.show()

    return predictions


# -----------------------------
# Final Prediction Loop
# -----------------------------
def predict_loop():

    uploaded = files.upload()

    for filename in uploaded.keys():

        predictions = predict_all_slices(
            filename
        )

        if not predictions:

            print(
                "Invalid Image. Please upload a valid Brain MRI image."
            )

            continue

        class_ids = [
            p[0] for p in predictions
        ]

        confidences = [
            p[1] for p in predictions
        ]

        final_confidence = max(
            confidences
        )

        # Priority Rule
        # Glioma > Other Tumor > Healthy

        if 1 in class_ids:

            print(
                f"\nGlioma Present (Confidence: {final_confidence:.2f})"
            )

        elif 2 in class_ids:

            print(
                f"\nOther Tumor Present (Confidence: {final_confidence:.2f})"
            )

        else:

            print(
                f"\nHealthy Brain (Confidence: {final_confidence:.2f})"
            )


# -----------------------------
# Run Prediction
# -----------------------------
predict_loop()

In [ ]:
# Save the model
model.save("brain_tumor_cnn.h5")

# Then download it
from google.colab import files
files.download("brain_tumor_cnn.h5")

In [ ]:
import matplotlib.pyplot as plt

labels = ['Training', 'Validation', 'Testing']
sizes = [49, 21, 30]

plt.figure(figsize=(6,6))
plt.pie(sizes, labels=labels, autopct='%1.0f%%')
plt.title('Dataset Distribution')
plt.show()

In [ ]:
import tensorflow as tf
print("TensorFlow Version:", tf.__version__)

In [ ]:
import sys
print("Python Version:", sys.version)

In [ ]:
import tensorflow as tf
print("TensorFlow Version:", tf.__version__)
print("Keras Version:", tf.keras.__version__)